# 🍎 Phi-4-reasoning Model with AIProjectClient 🍏

**Phi-4-reasoning** is an open-weight reasoning model from Microsoft (14B parameters) that is fine-tuned to think step by step, delivering strong results on math, coding, and STEM tasks while staying small enough to run cost-effectively.

In this notebook, you'll see how to:
1. **Initialize** an `AIProjectClient` for your Azure AI Foundry environment.
2. **Chat** with the **Phi-4-reasoning** model using the project's OpenAI client (`get_openai_client()`).
3. **Show** a Health & Fitness example, featuring disclaimers and wellness Q&A.
4. **Enjoy** strong step-by-step responses from a small, cost-effective model. 🏋️

> **Disclaimer**: This is not medical advice. Please consult professionals.

## Why Phi-4-reasoning?
- **Small but Capable**: 14B parameters deliver quality competitive with much larger models, at a fraction of the cost.
- **Reasoning-tuned**: Trained to work through problems step by step, which helps on math, planning, and logic.
- **Generous Context Window**: Enough room for multi-turn conversations.
- **Open Weight**: Available in the Foundry catalog for flexible deployment.

## 1. Setup

Below, we'll import the necessary libraries:
- **azure-ai-projects**: For the endpoint-based `AIProjectClient`.
- **openai** (via `project.get_openai_client()`): For chat completions against the deployed model.
- **azure-identity**: For `DefaultAzureCredential`.

Ensure you have a `.env` file with:
```bash
PROJECT_ENDPOINT=<your-project-endpoint>
MICROSOFT_MODEL=Phi-4-reasoning
```

> **Note**: It's recommended to complete the [`3-basic-rag.ipynb`](./3-basic-rag.ipynb) notebook before this one, as it covers important concepts that will be helpful here.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Phi-4-reasoning deployment name (from Build > Models in the Foundry portal)
phi4_deployment = os.getenv("MICROSOFT_MODEL", "Phi-4-reasoning")

try:
    # Endpoint-based client + its OpenAI client (used for chat completions)
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ AIProjectClient + OpenAI client created successfully!")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)

## 2. Chat with Phi-4-reasoning 🍏
We'll demonstrate a simple conversation using **Phi-4-reasoning** in a health & fitness context. We'll define a system prompt that clarifies the role of the assistant. Then we'll ask some user queries.

> Because this is a reasoning-tuned model, you can optionally ask it to show its work step-by-step, which is handy for math or planning questions.

In [ ]:
def chat_with_phi4(user_question, step_by_step=False):
    """Send a chat request to the Phi-4-reasoning model, optionally asking for step-by-step working."""
    # You'll define a system message with disclaimers:
    system_prompt = (
        "You are a Phi-4-reasoning AI assistant, focusing on health and fitness.\n"
        "Remind users that you are not a medical professional, but can provide general info.\n"
    )

    # You can optionally instruct the model to show its step-by-step working.
    if step_by_step:
        system_prompt += "Please show your step-by-step reasoning in your answer.\n"

    response = openai_client.chat.completions.create(
        model=phi4_deployment,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
        temperature=0.8,  # a bit creative
        top_p=0.9,
        max_tokens=400,
    )

    return response.choices[0].message.content

# Example usage:
question = "I'm training for a 5K. Any tips on a weekly workout schedule?"
answer = chat_with_phi4(question, step_by_step=True)
print("🗣️ User:", question)
print("🤖 Phi-4-reasoning:", answer)

## 3. RAG-like Example (Stub)
Phi-4-reasoning also works well in retrieval augmented generation scenarios, where you provide external context and let the model answer over it. Below is a **stub** example showing how you'd pass retrieved text as context.

> In a real scenario, you'd embed & search for relevant passages, then feed them into the user/system message.

In [ ]:
def chat_with_phi4_rag(user_question, retrieved_doc):
    """Simulate an RAG flow by appending retrieved context to the system prompt."""
    system_prompt = (
        "You are Phi-4-reasoning, a helpful fitness AI.\n"
        "We have some context from the user's knowledge base: \n"
        f"{retrieved_doc}\n"
        "Please use this context to help your answer. If the context doesn't help, say so.\n"
    )

    response = openai_client.chat.completions.create(
        model=phi4_deployment,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
        temperature=0.3,
        max_tokens=300,
    )
    return response.choices[0].message.content

# Define a dummy doc snippet:
doc_snippet = "Recommended to run 3 times per week and mix with cross-training.\n" \
              "Include rest days or active recovery days for muscle repair."

user_q = "How often should I run weekly to prepare for a 5K?"
rag_answer = chat_with_phi4_rag(user_q, doc_snippet)
print("🗣️ User:", user_q)
print("🤖 Phi-4-reasoning (RAG):", rag_answer)

## 4. Wrap-Up & Best Practices
1. **Step-by-Step Prompts**: Asking for explicit working can help on math or planning tasks — decide what to surface to end users.
2. **RAG**: Combine the OpenAI client with retrieval results to ground your answers.
3. **OpenTelemetry**: Optionally integrate `opentelemetry-sdk` and `azure-core-tracing-opentelemetry` for full observability.
4. **Evaluate**: Use `azure-ai-evaluation` to measure your model’s performance.
5. **Cost & Performance**: Phi-4 delivers strong results from a small 14B model at low cost. It shines on math/coding/STEM — evaluate for your domain needs.

## 🎉 Congratulations!
You've seen how to:
- Use **Phi-4** with `AIProjectClient` and its OpenAI client (`get_openai_client()`).
- Create a **chat** flow with an optional step-by-step prompt.
- Stub a **RAG** scenario.